# 01 — Ispezione dataset Emerging Jets (HDF5)

**Obiettivo del notebook:** fare la prima "fotografia" del dataset HDF5 fornito da ATLAS.
Capire la struttura, le shape, i dtype, vedere qualche valore concreto e calcolare statistiche di base sulle feature critiche.

**Cosa NON fa** (lo faremo nei notebook successivi):
- non disegna istogrammi (lo faremo in `02_eda_distribuzioni.ipynb`)
- non applica nessuna trasformazione ai dati
- non salva nulla su disco

**Pipeline** (8 sezioni):
1. Apertura del file e mappatura della struttura
2. Lettura degli attributi/metadata
3. Preview dei `jets`
4. Preview dei `tracks`
5. Statistiche di base sulle feature di jet
6. Verifica della maschera `valid` sulle tracce
7. Conteggio delle label (QCD vs EJ)
8. Statistiche sulle feature di traccia piu critiche

## 0. Setup

Importiamo le librerie e definiamo i parametri.

In [2]:
import subprocess
subprocess.run(["pip", "install", "h5py"])

CompletedProcess(args=['pip', 'install', 'h5py'], returncode=0)

In [3]:
import os
import numpy as np
import h5py

# Mostra tutto senza troncamenti quando stampiamo array piccoli
np.set_printoptions(precision=4, suppress=True, linewidth=120)

In [4]:
import os, platform, subprocess

print("Sistema:", platform.system(), platform.release())
print("Cartella corrente:", os.getcwd())
print()
print("Contenuto della cartella corrente:")
for item in sorted(os.listdir(".")):
    full = os.path.join(os.getcwd(), item)
    print(f"  [{'D' if os.path.isdir(full) else 'F'}] {item}")

print()
print("Vediamo se /mnt/c esiste (sintomo di WSL):")
print("  /mnt/c esiste:", os.path.exists("/mnt/c"))
if os.path.exists("/mnt/c/Users/delco/Desktop"):
    print("  Desktop trovato!")
    print("  Contenuto Desktop:")
    for x in sorted(os.listdir("/mnt/c/Users/delco/Desktop"))[:20]:
        print(f"    {x}")

Sistema: Windows 11
Cartella corrente: c:\Users\delco\Desktop\Università\Ad_M\dataset

Contenuto della cartella corrente:
  [D] .claude
  [D] .venv
  [F] 01_inspect_h5.ipynb
  [F] 01_inspect_h5.py
  [F] class_dict.yaml
  [F] ejs_test.yaml
  [F] ejs_train.yaml
  [F] ejs_val.yaml
  [F] norm_dict.yaml
  [F] pp_output_test_background.h5
  [F] pp_output_test_signal.h5
  [F] pp_output_val.h5

Vediamo se /mnt/c esiste (sintomo di WSL):
  /mnt/c esiste: False


In [5]:
# --- CONFIG -----------------------------------------------------------
# Partiamo dal file piu piccolo (signal di test, ~510 MB) per andare veloci.
# Per esplorare il file di validation cambia il path.
FILE_PATH = r"C:\Users\delco\Desktop\Università\Ad_M\dataset\pp_output_test_signal.h5"
#FILE_PATH = "/mnt/c/Users/delco/Desktop/Università/Ad_M/dataset/pp_output_test_signal.h5"

N_PREVIEW = 5             # righe da mostrare in preview
N_SAMPLE  = 200_000       # n. jets da usare per statistiche (None = tutti)

# Verifica che il file esista
assert os.path.exists(FILE_PATH), f"File non trovato: {FILE_PATH}"
print(f"File: {FILE_PATH}")
print(f"Dimensione: {os.path.getsize(FILE_PATH)/1e9:.2f} GB")

File: C:\Users\delco\Desktop\Università\Ad_M\dataset\pp_output_test_signal.h5
Dimensione: 0.51 GB


## 1. Struttura del file

Un file HDF5 e organizzato come un filesystem gerarchico: contiene **gruppi** (cartelle) e **dataset** (tensori). Vediamo cosa c'e dentro.

In [6]:
f = h5py.File(FILE_PATH, "r")   # NB: lo lasciamo aperto per tutto il notebook

def visit(name, obj):
    if isinstance(obj, h5py.Group):
        print(f"  [GROUP]   {name}")
    elif isinstance(obj, h5py.Dataset):
        print(f"  [DATASET] {name:30s} shape={obj.shape}  dtype={obj.dtype}")

print("STRUTTURA DEL FILE")
print("-" * 70)
f.visititems(visit)

STRUTTURA DEL FILE
----------------------------------------------------------------------
  [DATASET] jets                           shape=(61966,)  dtype=[('pt', '<f4'), ('eta', '<f4'), ('energy', '<f4'), ('mass', '<f4'), ('salt_pdisp', '<f4'), ('displacedPtFraction', '<f4'), ('Qw_clusterSoftDrop', '<f4'), ('C2_clusterSoftDrop', '<f4'), ('D2_clusterSoftDrop', '<f4'), ('Tau1_clusterSoftDrop', '<f4'), ('Tau2_clusterSoftDrop', '<f4'), ('Tau3_clusterSoftDrop', '<f4'), ('Tau4_clusterSoftDrop', '<f4'), ('Tau21_clusterSoftDrop', '<f4'), ('Tau32_clusterSoftDrop', '<f4'), ('ECF1_clusterSoftDrop', '<f4'), ('ECF2_clusterSoftDrop', '<f4'), ('ECF3_clusterSoftDrop', '<f4'), ('ECF4_clusterSoftDrop', '<f4'), ('Split12_clusterSoftDrop', '<f4'), ('Split23_clusterSoftDrop', '<f4'), ('Split34_clusterSoftDrop', '<f4'), ('timing_clusterSoftDrop', '<f4'), ('isTagged', '<i4'), ('isDisplaced', '<i4'), ('mcEventWeight', '<f4'), ('eventNumber', '<i8'), ('averageInteractionsPerCrossing', '<f4'), ('actualInteract

## 2. Attributi (metadata)

In HDF5 i dataset e i gruppi possono avere **attributi** (chiave-valore) che descrivono i dati (unita di misura, autore, versione del preprocessing, ...).

In [10]:
print("Attributi a livello ROOT:")
if len(f.attrs) == 0:
    print("  (nessuno)")
for k, v in f.attrs.items():
    print(f"  {k} = {v}")

for gname in f.keys():
    g = f[gname]
    if hasattr(g, "attrs") and len(g.attrs) > 0:
        print(f"\nAttributi di '{gname}':")
        for k, v in g.attrs.items():
            print(f"  {k} = {v}")

Attributi a livello ROOT:
  (nessuno)


## 3. Preview dei `jets`

Stampiamo le prime `N_PREVIEW` righe del dataset `jets`. Se e uno *structured array* (con campi nominati tipo `pt`, `eta`, ...) lo stampiamo in modo leggibile riga per riga.

In [8]:
jets = f["jets"]
print(f"Dataset 'jets': shape={jets.shape}, dtype={jets.dtype}")
field_names = jets.dtype.names
print(f"Campi disponibili: {field_names}")

Dataset 'jets': shape=(61966,), dtype=[('pt', '<f4'), ('eta', '<f4'), ('energy', '<f4'), ('mass', '<f4'), ('salt_pdisp', '<f4'), ('displacedPtFraction', '<f4'), ('Qw_clusterSoftDrop', '<f4'), ('C2_clusterSoftDrop', '<f4'), ('D2_clusterSoftDrop', '<f4'), ('Tau1_clusterSoftDrop', '<f4'), ('Tau2_clusterSoftDrop', '<f4'), ('Tau3_clusterSoftDrop', '<f4'), ('Tau4_clusterSoftDrop', '<f4'), ('Tau21_clusterSoftDrop', '<f4'), ('Tau32_clusterSoftDrop', '<f4'), ('ECF1_clusterSoftDrop', '<f4'), ('ECF2_clusterSoftDrop', '<f4'), ('ECF3_clusterSoftDrop', '<f4'), ('ECF4_clusterSoftDrop', '<f4'), ('Split12_clusterSoftDrop', '<f4'), ('Split23_clusterSoftDrop', '<f4'), ('Split34_clusterSoftDrop', '<f4'), ('timing_clusterSoftDrop', '<f4'), ('isTagged', '<i4'), ('isDisplaced', '<i4'), ('mcEventWeight', '<f4'), ('eventNumber', '<i8'), ('averageInteractionsPerCrossing', '<f4'), ('actualInteractionsPerCrossing', '<f4'), ('nPrimaryVertices', '<i4'), ('isRun3', 'i1')]
Campi disponibili: ('pt', 'eta', 'energy', '

In [11]:
preview = jets[:N_PREVIEW]
if field_names:
    for i, row in enumerate(preview):
        print(f"--- jet {i} ---")
        for name in field_names:
            print(f"  {name:30s} = {row[name]}")
else:
    print(preview)

--- jet 0 ---
  pt                             = 391629.6875
  eta                            = 1.1941876411437988
  energy                         = 711230.1875
  mass                           = 88660.3671875
  salt_pdisp                     = 1.0
  displacedPtFraction            = 0.9661736488342285
  Qw_clusterSoftDrop             = 35867.60546875
  C2_clusterSoftDrop             = 0.22978074848651886
  D2_clusterSoftDrop             = 1.7265828847885132
  Tau1_clusterSoftDrop           = 0.18716448545455933
  Tau2_clusterSoftDrop           = 0.11527643352746964
  Tau3_clusterSoftDrop           = 0.06391561776399612
  Tau4_clusterSoftDrop           = 0.04038872569799423
  Tau21_clusterSoftDrop          = 0.61590975522995
  Tau32_clusterSoftDrop          = 0.5544552206993103
  ECF1_clusterSoftDrop           = 399034.9375
  ECF2_clusterSoftDrop           = 21190834176.0
  ECF3_clusterSoftDrop           = 258582314483712.0
  ECF4_clusterSoftDrop           = 9.237531135280415e+17
  Spl

## 4. Preview dei `tracks`

Le tracce sono organizzate in un array 2D: `(N_jets, N_tracks_max)` con padding. Vediamo le prime 5 tracce del primo jet.

In [12]:
tracks = f["tracks"]
print(f"Dataset 'tracks': shape={tracks.shape}, dtype={tracks.dtype}")
track_fields = tracks.dtype.names
print(f"Numero di feature per traccia: {len(track_fields) if track_fields else 'array semplice'}")
print(f"Campi: {track_fields}")

Dataset 'tracks': shape=(61966, 200), dtype=[('numberOfInnermostPixelLayerHits', 'u1'), ('numberOfPixelHits', 'u1'), ('numberOfPixelHoles', 'u1'), ('numberOfPixelSharedHits', 'u1'), ('numberOfSCTHits', 'u1'), ('numberOfSCTHoles', 'u1'), ('numberOfSCTSharedHits', 'u1'), ('truthOriginLabel', '<i4'), ('truthVertexIndex', '<i4'), ('VSIVertexIndex', '<i4'), ('numberDoF', '<f2'), ('theta', '<f2'), ('qOverP', '<f4'), ('chiSquared', '<f4'), ('radiusOfFirstHit', '<f4'), ('pt_log1p', '<f4'), ('d0_log1p', '<f4'), ('pt', '<f4'), ('deta', '<f4'), ('z0RelativeToBeamspot', '<f4'), ('qOverPUncertainty', '<f4'), ('d0', '<f2'), ('z0SinTheta', '<f2'), ('eta', '<f2'), ('d0Uncertainty', '<f2'), ('z0SinThetaUncertainty', '<f2'), ('phiUncertainty', '<f2'), ('thetaUncertainty', '<f2'), ('dphi', '<f2'), ('dr', '<f2'), ('ptfrac', '<f2'), ('z0RelativeToBeamspotUncertainty', '<f2'), ('IP3D_signed_d0', '<f2'), ('IP2D_signed_d0', '<f2'), ('IP3D_signed_z0', '<f2'), ('IP3D_signed_d0_significance', '<f2'), ('IP3D_sign

In [13]:
# Prime 5 tracce del primo jet
if tracks.ndim == 2:
    first_jet_tracks = tracks[0, :5]
else:
    first_jet_tracks = tracks[:5]

if track_fields:
    for i, row in enumerate(first_jet_tracks):
        print(f"--- track {i} del primo jet ---")
        for name in track_fields:
            print(f"  {name:35s} = {row[name]}")
else:
    print(first_jet_tracks)

--- track 0 del primo jet ---
  numberOfInnermostPixelLayerHits     = 0
  numberOfPixelHits                   = 1
  numberOfPixelHoles                  = 0
  numberOfPixelSharedHits             = 0
  numberOfSCTHits                     = 8
  numberOfSCTHoles                    = 0
  numberOfSCTSharedHits               = 0
  truthOriginLabel                    = 8
  truthVertexIndex                    = 1
  VSIVertexIndex                      = -2
  numberDoF                           = 33.0
  theta                               = 0.88037109375
  qOverP                              = -0.00024995143758133054
  chiSquared                          = 28.760211944580078
  radiusOfFirstHit                    = 124.29618835449219
  pt_log1p                            = 8.034462928771973
  d0_log1p                            = 4.058089256286621
  pt                                  = 3084.4814453125
  deta                                = -0.44129636883735657
  z0RelativeToBeamspot             

## 5. Statistiche di base sui jets

Per ogni feature numerica dei jet (sample da `N_SAMPLE` esempi) calcoliamo:
- min, max, mean, std
- frazione di NaN e Inf (red flag se diversa da 0%)

In [14]:
n_jets_total = jets.shape[0]
n_sample = min(N_SAMPLE, n_jets_total) if N_SAMPLE else n_jets_total
print(f"N jet totali nel file: {n_jets_total}")
print(f"Sample usato: {n_sample}")

jets_sample = jets[:n_sample]

N jet totali nel file: 61966
Sample usato: 61966


In [15]:
print(f"{'feature':30s} {'min':>10s} {'max':>10s} {'mean':>10s} {'std':>10s}  NaN%   Inf%")
print("-" * 90)
for name in field_names:
    arr = jets_sample[name]
    if not np.issubdtype(arr.dtype, np.number):
        print(f"{name:30s}  (non numerica, dtype={arr.dtype})")
        continue
    nan_frac = np.isnan(arr).mean() if np.issubdtype(arr.dtype, np.floating) else 0.0
    inf_frac = np.isinf(arr).mean() if np.issubdtype(arr.dtype, np.floating) else 0.0
    print(f"{name:30s} {arr.min():10.3g} {arr.max():10.3g} "
          f"{arr.mean():10.3g} {arr.std():10.3g}  "
          f"{nan_frac:5.2%}  {inf_frac:5.2%}")

feature                               min        max       mean        std  NaN%   Inf%
------------------------------------------------------------------------------------------
pt                                  2e+05   1.62e+06   4.54e+05   1.48e+05  0.00%  0.00%
eta                                  -2.5        2.5    0.00863      0.996  0.00%  0.00%
energy                           2.05e+05   3.76e+06   7.25e+05    3.5e+05  0.00%  0.00%
mass                                  765   7.04e+05   1.59e+05   6.62e+04  0.00%  0.00%
salt_pdisp                       1.67e-05          1       0.96      0.181  0.00%  0.00%
displacedPtFraction                     0        5.6        1.2      0.427  0.00%  0.00%
Qw_clusterSoftDrop                      0      3e+05   6.68e+04   3.51e+04  0.00%  0.00%
C2_clusterSoftDrop                   -999      0.647     -0.423       27.2  0.00%  0.00%
D2_clusterSoftDrop                   -999       9.99       1.01       27.3  0.00%  0.00%
Tau1_clusterSoftDrop

c:\Users\delco\miniconda3\Lib\site-packages\numpy\_core\_methods.py:190: RuntimeWarning: overflow encountered in square
  x = um.square(x, out=x)
c:\Users\delco\miniconda3\Lib\site-packages\numpy\_core\_methods.py:201: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)


## 6. Maschera `valid` sulle tracce

Nel dataset ogni jet ha fino a `N_tracks_max` tracce, ma la maggior parte sono **padding**. Il campo `valid` (booleano) dice quali tracce sono reali. Vediamo la distribuzione del numero di tracce reali per jet.

In [16]:
if "valid" in track_fields:
    valid_sample = tracks["valid"][:n_sample]   # (n_sample, N_tracks_max)
    n_valid_per_jet = valid_sample.sum(axis=1)
    
    print(f"Padding (tracce max per jet): {tracks.shape[1]}")
    print(f"Tracce valide per jet:")
    print(f"  min     = {n_valid_per_jet.min()}")
    print(f"  max     = {n_valid_per_jet.max()}")
    print(f"  mean    = {n_valid_per_jet.mean():.1f}")
    print(f"  median  = {np.median(n_valid_per_jet):.0f}")
    print(f"  P95     = {np.percentile(n_valid_per_jet, 95):.0f}")
    print(f"  P99     = {np.percentile(n_valid_per_jet, 99):.0f}")
    print()
    print(f"Jet con 0 tracce valide: {(n_valid_per_jet == 0).sum()} "
          f"({(n_valid_per_jet == 0).mean():.2%})")
else:
    print("Campo 'valid' non trovato nel gruppo tracks")

Padding (tracce max per jet): 200
Tracce valide per jet:
  min     = 0
  max     = 200
  mean    = 50.4
  median  = 46
  P95     = 96
  P99     = 125

Jet con 0 tracce valide: 1 (0.00%)


## 7. Bilanciamento delle label

Contiamo quanti jet QCD (label 0) e quanti EJ (label 1) ci sono nel file. Le label da controllare sono `isDisplaced` e `flavour_label`.

In [17]:
for lab in ["isDisplaced", "flavour_label"]:
    if lab in field_names:
        arr = jets_sample[lab]
        vals, counts = np.unique(arr, return_counts=True)
        print(f"Label '{lab}':")
        for v, c in zip(vals, counts):
            print(f"  {v}: {c}  ({c/len(arr):.2%})")
        print()
    else:
        print(f"Label '{lab}' non presente nel dataset 'jets'")

Label 'isDisplaced':
  0: 2322  (3.75%)
  1: 59644  (96.25%)

Label 'flavour_label' non presente nel dataset 'jets'


## 8. Statistiche sulle feature di traccia critiche

Per le 8 feature di traccia piu importanti calcoliamo statistiche **solo sulle tracce valide** (non sul padding). Includiamo anche i percentili 1% e 99% per vedere quanto sono "estreme" le code.

In [18]:
CRITICAL_TRACK_FEATURES = [
    "d0",
    "z0SinTheta",
    "qOverP",
    "pt",
    "IP3D_signed_d0_significance",
    "numberOfInnermostPixelLayerHits",
    "pt_log1p",
    "d0_log1p",
]

N_TRACK_SAMPLE = min(20_000, n_jets_total)
print(f"Sample: primi {N_TRACK_SAMPLE} jets (~ {N_TRACK_SAMPLE * tracks.shape[1]:.0e} tracce con padding)")
print()

tracks_sample = tracks[:N_TRACK_SAMPLE]
valid_mask = tracks_sample["valid"].astype(bool) if "valid" in track_fields else None

Sample: primi 20000 jets (~ 4e+06 tracce con padding)



In [19]:
print(f"{'feature':40s} {'min':>10s} {'max':>10s} {'p1':>10s} {'p99':>10s} {'mean':>10s} {'std':>10s}")
print("-" * 100)
for name in CRITICAL_TRACK_FEATURES:
    if name not in track_fields:
        print(f"{name:40s}  (non presente)")
        continue
    arr = tracks_sample[name]
    if valid_mask is not None:
        arr = arr[valid_mask]
    if np.issubdtype(arr.dtype, np.floating):
        arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        print(f"{name:40s}  (vuoto)")
        continue
    p1, p99 = np.percentile(arr, [1, 99])
    print(f"{name:40s} {arr.min():10.3g} {arr.max():10.3g} "
          f"{p1:10.3g} {p99:10.3g} {arr.mean():10.3g} {arr.std():10.3g}")

feature                                         min        max         p1        p99       mean        std
----------------------------------------------------------------------------------------------------
d0                                             -300        300       -201        186     -0.688        inf
z0SinTheta                                     -500        498       -270        271     -0.124        inf
qOverP                                       -0.001      0.001  -0.000907    0.00091   6.03e-06   0.000442


c:\Users\delco\miniconda3\Lib\site-packages\numpy\_core\_methods.py:168: RuntimeWarning: overflow encountered in reduce
  arrmean = umr_sum(arr, axis, dtype, keepdims=True, where=where)


pt                                            1e+03   7.25e+08   1.01e+03   7.01e+04   1.03e+04   8.45e+05
IP3D_signed_d0_significance                -1.5e+04   1.36e+04       -640        719       10.4        inf
numberOfInnermostPixelLayerHits                   0          2          0          2      0.444      0.547
pt_log1p                                       6.91       20.4       6.92       11.2        7.9      0.961
d0_log1p                                      -5.71       5.71      -5.31       5.23     -0.053       2.21


## Chiusura del file

Importante: HDF5 tiene il file aperto, ricordiamoci sempre di chiuderlo a fine sessione.

In [20]:
f.close()
print("File chiuso.")

File chiuso.


---

## Cosa fare ora con l'output di questo notebook

1. Verifica che le **shape** dei dataset siano coerenti (n. jet uguale in `jets` e `tracks`)
2. Controlla la **distribuzione del numero di tracce per jet**: il P99 ti dice il padding sensato
3. Controlla **NaN e Inf**: se diversi da 0% bisogna gestirli
4. Controlla che `isDisplaced` abbia valori sensati (0 per QCD, 1 per EJ) e che il file di signal contenga effettivamente solo segnale (o anche un po' di QCD per controllo)
5. Confronta le statistiche delle feature di traccia con i valori di `norm_dict.yaml`: devono essere coerenti

Quando hai l'output, condividilo e passiamo al notebook successivo (**02 — distribuzioni e istogrammi**) dove vedremo finalmente la forma vera delle feature.